# Economic-Aware AI — Expected Loss Minimization

**HAIIP Phase 6 — Research Notebook**

This notebook demonstrates the Economic Decision Engine:
- Expected cost formulas
- Decision boundary visualisation
- Fleet ROI analysis
- Comparison: naive threshold vs expected-loss decision

**References**:
- Pintelon & Parodi-Herz (2008) Maintenance Decision Making
- Mobley (2002) An Introduction to Predictive Maintenance

> **Experimental**: This is a research deliverable. Do not use in production without validation.

In [ ]:
import sys
sys.path.insert(0, '..')  # repo root

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

from haiip.core.economic_ai import CostProfile, EconomicDecisionEngine, MaintenanceAction

print('HAIIP Economic AI ready')

## 1. Cost Profile — Nordic SME defaults

In [ ]:
profile = CostProfile(
    production_rate_eur_hr=500.0,  # €500/hr production revenue
    downtime_hours_avg=8.0,        # 8h average downtime per failure
    labour_rate_eur_hr=85.0,
    labour_hours_avg=4.0,
    parts_cost_eur=250.0,
    opportunity_cost_eur=150.0,
    safety_factor=1.5,
)

print(f'C_downtime:        €{profile.c_downtime:,.0f}')
print(f'C_maintenance:     €{profile.c_maintenance:,.0f}')
print(f'C_false_negative:  €{profile.c_false_negative:,.0f}')
print(f'C_false_positive:  €{profile.c_false_positive:,.0f}')
print(f'Break-even P(f):   {profile.c_maintenance / profile.c_false_negative:.3f}')

## 2. Decision Boundary Visualisation

In [ ]:
engine = EconomicDecisionEngine(cost_profile=profile)

p_range     = np.linspace(0, 1, 100)
score_range = np.linspace(0, 1, 100)
P, S = np.meshgrid(p_range, score_range)

action_map = {
    MaintenanceAction.REPAIR_NOW: 3,
    MaintenanceAction.SCHEDULE:   2,
    MaintenanceAction.MONITOR:    1,
    MaintenanceAction.IGNORE:     0,
}

Z = np.zeros_like(P)
for i in range(P.shape[0]):
    for j in range(P.shape[1]):
        d = engine.decide(anomaly_score=S[i,j], failure_probability=P[i,j])
        Z[i,j] = action_map[d.action]

fig, ax = plt.subplots(figsize=(10, 7))
cmap = plt.cm.get_cmap('RdYlGn_r', 4)
im = ax.contourf(P, S, Z, levels=[-0.5, 0.5, 1.5, 2.5, 3.5], cmap=cmap)
cbar = plt.colorbar(im, ax=ax, ticks=[0, 1, 2, 3])
cbar.set_ticklabels(['IGNORE', 'MONITOR', 'SCHEDULE', 'REPAIR NOW'])
ax.set_xlabel('P(failure)', fontsize=13)
ax.set_ylabel('Anomaly Score', fontsize=13)
ax.set_title('Economic Decision Boundary\n(C_downtime=€4,000, C_maintenance=€590)', fontsize=14)
plt.tight_layout()
plt.savefig('../docs/figures/economic_decision_boundary.png', dpi=150)
plt.show()

## 3. Fleet ROI Simulation — 100 machines × 30 days

In [ ]:
rng = np.random.default_rng(42)
n_machines = 100
n_days     = 30

# Simulate daily sensor readings for 100 machines
records = [
    {
        'anomaly_score':       float(rng.uniform(0, 0.95)),
        'failure_probability': float(rng.beta(1, 5)),  # skewed low (most machines fine)
        'machine_id':         f'M-{i:03d}',
    }
    for i in range(n_machines * n_days)
]

decisions = engine.batch_decide(records)
roi = engine.roi_summary(decisions)

print('=== 30-Day Fleet ROI ===')
print(f"Total net benefit:           €{roi['total_net_benefit']:>12,.2f}")
print(f"Projected downtime savings:  €{roi['projected_downtime_savings_eur']:>12,.2f}")
print(f"Human reviews triggered:     {roi['human_review_count']:>12}")
print(f"Decision breakdown:          {roi['decisions_by_action']}")

# Bar chart
fig, ax = plt.subplots(figsize=(8, 5))
actions = list(roi['decisions_by_action'].keys())
counts  = list(roi['decisions_by_action'].values())
colors  = ['#d32f2f', '#f57c00', '#388e3c', '#1976d2']
ax.bar(actions, counts, color=colors[:len(actions)])
ax.set_xlabel('Decision', fontsize=12)
ax.set_ylabel('Count', fontsize=12)
ax.set_title(f'Fleet Decisions — {n_machines} machines × {n_days} days', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Naive Threshold vs Expected Loss — F1 Comparison

In [ ]:
from sklearn.metrics import f1_score

rng = np.random.default_rng(0)
n = 500
true_failure = rng.random(n) < 0.15  # 15% failure rate
anomaly_scores = np.where(true_failure, rng.beta(5, 2, n), rng.beta(1, 5, n))
p_failure = np.where(true_failure, rng.beta(4, 2, n), rng.beta(1, 6, n))

# Naive: threshold at 0.5
naive_pred = (p_failure > 0.5).astype(int)
naive_f1   = f1_score(true_failure, naive_pred)

# Economic: REPAIR_NOW or SCHEDULE → predicted failure
econ_pred = np.array([
    1 if engine.decide(s, p).action in (MaintenanceAction.REPAIR_NOW, MaintenanceAction.SCHEDULE)
    else 0
    for s, p in zip(anomaly_scores, p_failure)
])
econ_f1 = f1_score(true_failure, econ_pred)

print(f'Naive threshold F1:    {naive_f1:.4f}')
print(f'Economic decision F1:  {econ_f1:.4f}')
print(f'Improvement:           {econ_f1 - naive_f1:+.4f}')